# scikit-learn pipelines: one leakage-safe object

Machine Learning Engineer track, lesson MLE1. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** turn the churn model into one object that takes raw rows and gives chances, know what it accepts and refuses at serving time, keep every learned number inside its steps, and save and reload it.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import json
import math
import numbers
import os
import subprocess
import sys
import tempfile
import urllib.request

import joblib
import numpy as np
import pandas as pd

try:
    import duckdb
except ImportError:  # Colab usually has DuckDB; install it when it does not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb
import sklearn
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

SEED = 2026       # the seed of the whole track

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.

RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong, tol):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"      # the same files, when this notebook runs inside a clone of the repository


def fetch(name):
    """The path of one CSV file: downloaded from the tag, or the local copy."""
    try:
        path = os.path.join(tempfile.mkdtemp(), name + ".csv")
        urllib.request.urlretrieve(BASE + name + ".csv", path)
        return path
    except Exception:
        return LOCAL + name + ".csv"

con = duckdb.connect()                       # a new, empty database every time this cell runs
con.execute(f"CREATE TABLE users AS FROM read_csv('{fetch('users')}', types={{'signup_date': 'DATE'}})")
con.execute(f"CREATE TABLE events AS FROM read_csv('{fetch('events')}', types={{'event_ts': 'TIMESTAMP', 'received_at': 'TIMESTAMP'}})")

# Known answer from the dataset README: 21,841 people and 177,379 events.
assert con.execute("SELECT (SELECT COUNT(*) FROM users), (SELECT COUNT(*) FROM events)").fetchall()[0] == (21841, 177379)

# The modeling table of the Data Scientist track: one row per person, built by the query of lesson DS3, "Building a modeling table in SQL".
# Run this cell as it is; DS3 explains it line by line.
MODELING_SQL = """WITH people AS (            -- one row per person: the spine of the table
  SELECT u.user_id, u.signup_date, u.channel, u.region, s.platform,
         s.event_ts AS signup_ts,
         s.event_ts + INTERVAL 7 DAY AS cutoff,       -- the prediction moment
         s.event_ts + INTERVAL 30 DAY AS horizon      -- the end of the label window
  FROM users AS u
  JOIN events AS s ON s.user_id = u.user_id AND s.event_name = 'signup_complete'
  WHERE u.signup_date BETWEEN DATE '2026-01-01' AND DATE '2026-05-31'
),
clean AS (                  -- retry copies removed: keep the first copy of each event
  SELECT *
  FROM events
  QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id, event_ts, event_name, platform ORDER BY received_at, event_id) = 1
),
known AS (                  -- what the app knew at the cutoff: it had happened and it had arrived
  SELECT p.user_id, c.event_name, c.event_ts, p.cutoff
  FROM people AS p
  JOIN clean AS c ON c.user_id = p.user_id
  WHERE c.event_ts <= p.cutoff AND c.received_at <= p.cutoff
),
features AS (
  SELECT user_id,
         COUNT(*) FILTER (WHERE event_name = 'session_start') AS sessions_7d,
         COUNT(*) FILTER (WHERE event_name = 'add_expense') AS expenses_7d,
         COUNT(*) FILTER (WHERE event_name IN ('session_start', 'add_expense') AND event_ts > cutoff - INTERVAL 3 DAY) AS recent_3d,
         COUNT(DISTINCT CAST(event_ts AS DATE)) FILTER (WHERE event_name IN ('session_start', 'add_expense')) AS active_days_7d,
         CAST(COUNT(*) FILTER (WHERE event_name = 'onboarding_complete') > 0 AS INTEGER) AS onboarded,
         CAST(COUNT(*) FILTER (WHERE event_name IN ('view_paywall', 'paywall_viewed')) > 0 AS INTEGER) AS saw_paywall,
         CAST(COUNT(*) FILTER (WHERE event_name = 'purchase') > 0 AS INTEGER) AS bought_plus
  FROM known
  GROUP BY user_id
),
label AS (                  -- from the other side of the cutoff: no visit and no expense in days 8 to 30
  SELECT p.user_id,
         CAST(NOT EXISTS (SELECT 1 FROM events AS e
                          WHERE e.user_id = p.user_id AND e.event_name IN ('session_start', 'add_expense')
                            AND e.event_ts > p.cutoff AND e.event_ts <= p.horizon) AS INTEGER) AS churned
  FROM people AS p
)
SELECT p.user_id, p.signup_date, p.channel, p.platform, p.region,
       f.sessions_7d, f.expenses_7d, f.recent_3d, f.active_days_7d, f.onboarded, f.saw_paywall, f.bought_plus,
       l.churned
FROM people AS p
JOIN features AS f ON f.user_id = p.user_id
JOIN label AS l ON l.user_id = p.user_id
ORDER BY p.user_id"""
table = con.sql(MODELING_SQL).df()
print(f"Data loaded: {len(table):,} people, {int(table['churned'].sum()):,} of them churned.")

## The split and the columns

The columns, the split and the five folds of lesson DS5, the same ones every model of the Data Scientist track used. The test part is not scored in this lesson: its rows stand in for requests that arrive after the model is built. Run this cell as it is.

In [ ]:
STATIC = ["channel", "platform", "region"]                    # known at signup
WEEK1 = ["sessions_7d", "expenses_7d", "recent_3d", "active_days_7d", "onboarded", "saw_paywall", "bought_plus"]    # known at day 7
X, y = table[STATIC + WEEK1], table["churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)
folds = StratifiedKFold(5, shuffle=True, random_state=SEED)       # the five folds of lesson DS5
print(len(X_train), len(X_test))

## Unit 1: One object from raw rows to chances

**Predict first.** The table has 10 columns, 3 of them text. How many columns will the logistic regression receive, and how many of those come from the text columns? And will the cross-validated mean AUC of this pipeline be lower than, equal to or higher than the 0.692 of lesson DS5?

In [ ]:
cat_columns = make_column_selector(dtype_exclude="number")     # every text column, picked by its type, not by its name
num_columns = make_column_selector(dtype_include="number")     # every number column
prep = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat_columns),
                          ("num", StandardScaler(), num_columns)])
pipe = Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=1000))])
pipe.fit(X_train, y_train)                                     # raw rows in: the pipeline fits each step in order
names = pipe[:-1].get_feature_names_out()                      # pipe[:-1] is every step except the model
print(len(names), sum(name.startswith("cat__") for name in names))
scores = cross_val_score(pipe, X_train, y_train, cv=folds, scoring="roc_auc")
print(round(float(scores.mean()), 3))

## Unit 2: What it accepts and what it refuses

**Predict first.** Three requests are scored by the fitted pipeline: the same five people with their columns in the opposite order, the same five with a region the training people never had, and the same five with one number missing. Which will be scored and which will raise an error?

In [ ]:
five = X_test.iloc[:5]                                         # five people, as if they were five requests
base = pipe.predict_proba(five)[:, 1]                          # the chance of churning, column 1 of the two chances
requests = {"columns backwards": five[five.columns[::-1]],
            "a region never seen": five.assign(region="africa"),
            "a missing number": five.assign(sessions_7d=np.nan)}
for label, rows in requests.items():
    try:
        chances = pipe.predict_proba(rows)[:, 1]
        print(f"{label}: scored, biggest change from base {np.abs(chances - base).max():.3f}")
    except ValueError as error:
        print(f"{label}: ValueError, {str(error).splitlines()[0]}")

## Unit 3: What a step learns, it learns in fit

**Predict first.** Two pipelines differ in one step: the first centers each number column with a mean learned in `fit` (`StandardScaler(with_std=False)`), the second subtracts the mean of whatever rows it is given. Will cross-validation tell them apart? And for the first five test people, will the chances be the same scored all 4,454 together as scored one at a time?

In [ ]:
def center_batch(rows):
    """A function step: subtract the mean of the rows it is given."""
    return rows - rows.mean()


def build(number_step):
    prep = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat_columns), ("num", number_step, num_columns)])
    return Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=1000))])


pipelines = {"learned": build(StandardScaler(with_std=False)),       # keeps each column's training mean after fit
             "recomputed": build(FunctionTransformer(center_batch))}    # a function: nothing is kept
for label, candidate in pipelines.items():
    cv = cross_val_score(candidate, X_train, y_train, cv=folds, scoring="roc_auc")
    candidate.fit(X_train, y_train)
    together = candidate.predict_proba(X_test)[:, 1][:5]
    alone = np.array([candidate.predict_proba(X_test.iloc[[i]])[:, 1][0] for i in range(5)])
    print(label, round(float(cv.mean()), 3), round(float(np.abs(together - alone).max()), 3))

## Unit 4: Save it, reload it

**Predict first.** The fitted pipeline of unit 1 is written to a file with `joblib.dump` and read back with `joblib.load`. Will the reloaded pipeline give exactly the same 4,454 chances as the one in memory? And will the file be smaller than 10 KB (10,000 bytes)?

In [ ]:
path = os.path.join(tempfile.mkdtemp(), "churn_pipeline.joblib")      # a file in a throwaway folder
joblib.dump(pipe, path)                                       # write the fitted pipeline: every step, with what it learned
loaded = joblib.load(path)                                    # read it back: no fit, no training rows
print(np.array_equal(pipe.predict_proba(X_test), loaded.predict_proba(X_test)))
print(os.path.getsize(path) < 10_000)

Now the same file with a note beside it. The helpers below are given; run this cell as it is.

In [ ]:
def save_pipeline(pipeline, path):
    """Write the pipeline, and beside it a note of the library version and the columns it expects."""
    joblib.dump(pipeline, path)
    with open(path + ".json", "w") as f:
        json.dump({"scikit_learn": sklearn.__version__, "columns": list(X_train.columns)}, f)


def load_pipeline(path):
    """Read the pipeline back, but refuse a file written by another scikit-learn version."""
    with open(path + ".json") as f:
        notes = json.load(f)
    if notes["scikit_learn"] != sklearn.__version__:
        raise RuntimeError(f"saved with scikit-learn {notes['scikit_learn']}, this is {sklearn.__version__}")
    return joblib.load(path)

**Try it:** save the pipeline with its note, read the note, change the version in it, and load again.

In [ ]:
save_pipeline(pipe, path)
with open(path + ".json") as f:
    notes = json.load(f)
print(sorted(notes))
notes["scikit_learn"] = "0.24.2"                              # pretend the file came from an older environment
with open(path + ".json", "w") as f:
    json.dump(notes, f)
try:
    load_pipeline(path)
except RuntimeError as error:
    print("refused:", str(error).split(",")[0])

## Finish it

### Exercise 1: Finish it

**Task:** **Score one request.** A request reaches the service as a dictionary, one value for each column (`request`, given). Turn it into the one-row table the pipeline expects and put the chance of churning, rounded to 3 decimals, in `answer_1`. Use the fitted `pipe` of unit 1.

**Expected:** `answer_1` is a number between 0 and 1.

In [ ]:
request = {"channel": "paid_social", "platform": "ios", "region": "europe", "sessions_7d": 2, "expenses_7d": 1,
           "recent_3d": 0, "active_days_7d": 1, "onboarded": 0, "saw_paywall": 1, "bought_plus": 0}      # given: one request
# make a one-row table from the dictionary, then ask the fitted pipe for the chance of churning
answer_1 = None  # replace None with the chance, rounded to 3 decimals

In [ ]:
_ = check("Exercise 1", "answer_1", 0.771, traps={1.0: "That is the class `predict` gives. Ask for the chance with `predict_proba` and take column 1."}, tol=5e-4)

<details>
<summary>Hint</summary>

A table is built from a list of rows. A dictionary is one row, so it goes inside a list: `pd.DataFrame([request])`.

</details>

<details>
<summary>Half-solution</summary>

```python
row = pd.DataFrame([...])
answer_1 = round(float(pipe.predict_proba(row)[:, 1][0]), 3)
```

</details>

<details>
<summary>Solution</summary>

```python
row = pd.DataFrame([request])
answer_1 = round(float(pipe.predict_proba(row)[:, 1][0]), 3)
```

The chance is 0.771. `pd.DataFrame(request)` raises an error, because every value is a single number and the table has no index; `[request]` makes it one row.

</details>

## On your own

### Exercise 2: On your own

**Task:** **A step that learns a cap.** The team wants `expenses_7d` capped at the 99th percentile of the people the model learned from, so one person with 34 expenses cannot stretch a column. Before you start, say in one sentence what a request with 500 expenses should be capped to. Write a class `CapExpenses` with a `fit` that learns the cap from the rows it is given (`np.percentile(rows["expenses_7d"], 99)`) and a `transform` that returns a copy of the rows with the column clipped to it. Fit it on `X_train` and put `[cap, capped]` in `answer_2`: the cap it learned and the value `transform` returns for a one-row table with 500 expenses. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a list of two numbers.

In [ ]:
_ = check("Exercise 2", "answer_2", [11.0, 11.0], traps={(500.0, 500.0): "The cap came from the request itself. Learn it in fit and keep it on the object."}, tol=5e-4)

<details>
<summary>Hint</summary>

A class with `fit(self, rows, y=None)` that stores `self.cap_` and returns `self`, and `transform(self, rows)` that uses `self.cap_`. Inherit from `BaseEstimator` and `TransformerMixin` (from `sklearn.base`) so that `clone` and a pipeline accept it.

</details>

<details>
<summary>Half-solution</summary>

```python
class CapExpenses(BaseEstimator, TransformerMixin):
    def fit(self, rows, y=None):
        self.cap_ = ...
        return self
```

</details>

<details>
<summary>Solution</summary>

```python
from sklearn.base import BaseEstimator, TransformerMixin


class CapExpenses(BaseEstimator, TransformerMixin):
    def fit(self, rows, y=None):
        self.cap_ = float(np.percentile(rows["expenses_7d"], 99))      # learned from the rows given to fit, kept on the object
        return self

    def transform(self, rows):
        rows = rows.copy()
        rows["expenses_7d"] = rows["expenses_7d"].clip(upper=self.cap_)
        return rows


capper = CapExpenses().fit(X_train)
one_row = pd.DataFrame({"expenses_7d": [500]})
answer_2 = [capper.cap_, float(capper.transform(one_row)["expenses_7d"].iloc[0])]
```

The cap is 11.0 expenses and a request with 500 expenses comes back as 11.0. A version that took the percentile inside `transform` would cap a one-row request to its own 500, so the cap would differ between training and serving.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** **Spot the flaw.** A colleague writes the target as words, `"left"` and `"stayed"`, fits the pipeline on them, and serves the chance of leaving as column 1 of `predict_proba`, as in the units. Change the last line so that `answer_3` holds the chances of leaving of the people in rows 5 to 9 of the test part, each rounded to 3 decimals.

**Expected:** `answer_3` is a list of five numbers between 0 and 1.

In [ ]:
words = y_train.map({1: "left", 0: "stayed"})                  # given: the target as words
words_pipe = build(StandardScaler()).fit(X_train, words)
requests3 = X_test.iloc[5:10]                                  # given: five requests
answer_3 = [round(float(c), 3) for c in words_pipe.predict_proba(requests3)[:, 1]]     # colleague: "column 1 is the chance of leaving"
# change the last line so that it holds the chance of leaving

In [ ]:
_ = check("Exercise 3", "answer_3", [0.403, 0.821, 0.586, 0.603, 0.548], traps=[([0.597, 0.179, 0.414, 0.397, 0.452], "These are the chances of staying: column 1 of `predict_proba` is the second class in `classes_`. Look the class up by name.")], tol=5e-4)

<details>
<summary>Hint</summary>

`predict_proba` has one column for each class. Which class is column 1 now? The pipeline lists them in `classes_`.

</details>

<details>
<summary>Half-solution</summary>

```python
position = list(words_pipe.classes_).index(...)
answer_3 = [round(float(c), 3) for c in words_pipe.predict_proba(requests3)[:, position]]
```

</details>

<details>
<summary>Solution</summary>

```python
position = list(words_pipe.classes_).index("left")
answer_3 = [round(float(c), 3) for c in words_pipe.predict_proba(requests3)[:, position]]
```

The columns follow the order of `classes_`, which is sorted: `['left', 'stayed']`. Column 1 was the chance of staying, so the colleague's numbers are one minus the chance of leaving. Looking up the position of the class by name keeps the answer right whatever the labels are.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")